# PySpark + Delta Lake — Local Development

Stack: **Spark 3.5.4 · Delta Lake 3.3.0 · Python 3.12**

Topics covered:
1. SparkSession setup
2. Write & read a Delta table
3. Schema enforcement & evolution
4. MERGE (upsert)
5. Time travel
6. OPTIMIZE & VACUUM
7. Delta SQL

In [ ]:
import pyspark
from delta import *

builder = pyspark.sql.SparkSession.builder.appName("MyApp") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-02935c05-0e31-40f0-97d1-e36dcc141cf9;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 208ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

16:32:58 WARN  NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Spark version : 3.5.4
Spark UI      : http://localhost:4040
16:33:13 WARN  GarbageCollectionMetrics - To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 1 · Write a Delta table

TABLE_PATH = '/workspace/delta-tables/users'

schema = StructType([
    StructField('id',         IntegerType(), False),
    StructField('name',       StringType(),  False),
    StructField('email',      StringType(),  True),
    StructField('created_at', TimestampType(), True),
])

data = [
    (1, 'Alice',   'alice@example.com',   None),
    (2, 'Bob',     'bob@example.com',     None),
    (3, 'Charlie', 'charlie@example.com', None),
]

df = spark.createDataFrame(data, schema).withColumn('created_at', F.current_timestamp())

(
    df.write
    .format('delta')
    .mode('overwrite')
    .save(TABLE_PATH)
)

spark.read.format('delta').load(TABLE_PATH).show()

## 2 · Append data (version 1)

In [ ]:
new_users = spark.createDataFrame([
    (4, 'Diana', 'diana@example.com', None),
    (5, 'Eve',   'eve@example.com',   None),
], schema).withColumn('created_at', F.current_timestamp())

new_users.write.format('delta').mode('append').save(TABLE_PATH)

spark.read.format('delta').load(TABLE_PATH).orderBy('id').show()

## 3 · MERGE (upsert)

In [ ]:
updates = spark.createDataFrame([
    (2, 'Bobby',  'bobby@example.com',  None),   # update existing
    (6, 'Frank',  'frank@example.com',  None),   # insert new
], schema).withColumn('created_at', F.current_timestamp())

target = DeltaTable.forPath(spark, TABLE_PATH)

(
    target.alias('t')
    .merge(updates.alias('s'), 't.id = s.id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

spark.read.format('delta').load(TABLE_PATH).orderBy('id').show()

## 4 · Time Travel

In [ ]:
# Read version 0 (original write)
v0 = spark.read.format('delta').option('versionAsOf', 0).load(TABLE_PATH)
print('Version 0 row count:', v0.count())
v0.show()

# View full history
DeltaTable.forPath(spark, TABLE_PATH).history().select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

## 5 · Schema Evolution

In [ ]:
# Add a new column via mergeSchema
with_status = spark.createDataFrame([
    (7, 'Grace', 'grace@example.com', None, 'active'),
], ['id', 'name', 'email', 'created_at', 'status']) \
    .withColumn('created_at', F.current_timestamp())

(
    with_status.write
    .format('delta')
    .mode('append')
    .option('mergeSchema', 'true')
    .save(TABLE_PATH)
)

spark.read.format('delta').load(TABLE_PATH).orderBy('id').show()

## 6 · Delta SQL

In [ ]:
spark.sql(f"CREATE TABLE IF NOT EXISTS users USING DELTA LOCATION '{TABLE_PATH}'")

spark.sql("SELECT id, name, status FROM users WHERE status IS NOT NULL").show()

spark.sql("DESCRIBE HISTORY users").select('version', 'timestamp', 'operation').show()

## 7 · OPTIMIZE & VACUUM

In [ ]:
# OPTIMIZE compacts small files
DeltaTable.forPath(spark, TABLE_PATH).optimize().executeCompaction()

# VACUUM removes files older than retention period (0 hours for demo — disable safety check)
spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled', 'false')
DeltaTable.forPath(spark, TABLE_PATH).vacuum(0)

print('Done. Final table:')
spark.read.format('delta').load(TABLE_PATH).orderBy('id').show()

In [ ]:
spark.stop()